# 🔄 Transformer Tutorial - Language Understanding & RUL Prediction

## Learning Objectives

By the end of this tutorial, you will:
- Understand Transformer architecture and attention mechanism
- Learn how self-attention works
- Build a Transformer model for language tasks (sentiment analysis)
- Apply Transformers to time series RUL prediction
- Visualize attention mechanisms
- Compare Transformer with LSTM/RNN

---

## What is a Transformer?

**Transformers** revolutionized deep learning by introducing the **attention mechanism**, eliminating the need for recurrence or convolution.

### Key Innovation: Self-Attention

- **Attention**: Model learns which parts of input are important
- **Self-Attention**: Each position can attend to all other positions
- **Parallel Processing**: No sequential dependency like RNNs
- **Long-range Dependencies**: Can capture relationships across entire sequence

### Why Transformers?

- **Efficiency**: Parallel processing (faster than RNNs)
- **Long-range Context**: Better at capturing distant dependencies
- **Flexibility**: Works for both language and time series
- **State-of-the-art**: Powers GPT, BERT, and modern AI

### Our Goals:

1. **Language Task**: Sentiment analysis (easy to understand!)
2. **RUL Prediction**: Apply transformers to engine degradation data

---

## Step 1: Import Required Libraries

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch, FancyBboxPatch, Arrow
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras for Transformers
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Embedding, Dropout, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# For transformer components
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Add

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Visualization style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')

sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Visualize Transformer Architecture

Let's understand how Transformers work!

In [ ]:
# Visualize Transformer Architecture
fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(-1, 12)
ax.set_ylim(-1, 10)
ax.axis('off')
ax.set_title('Transformer Architecture Overview', fontsize=18, fontweight='bold', pad=25)

# Encoder Stack (left)
encoder_y = 5
encoder_width = 2.5
encoder_height = 3

# Input Embeddings
rect_emb = Rectangle((0, encoder_y - 0.5), 2, 1, facecolor='#3498DB', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect_emb)
ax.text(1, encoder_y, 'Input\\nEmbeddings', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Positional Encoding
rect_pos = Rectangle((0, encoder_y - 1.8), 2, 1, facecolor='#9B59B6', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect_pos)
ax.text(1, encoder_y - 1.3, 'Positional\\nEncoding', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Encoder Layers (stacked)
for i in range(2):
    y_pos = encoder_y + 1.5 + i * 1.8
    rect_enc = Rectangle((0, y_pos - 0.8), 2, 1.6, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rect_enc)
    
    # Self-Attention
    rect_att = Rectangle((0.2, y_pos - 0.6), 0.7, 0.4, facecolor='#E74C3C', edgecolor='black', linewidth=1.5, alpha=0.7)
    ax.add_patch(rect_att)
    ax.text(0.55, y_pos - 0.4, 'Self-\\nAttention', ha='center', va='center', fontsize=7, fontweight='bold', color='white')
    
    # Feed Forward
    rect_ff = Rectangle((0.2, y_pos + 0.1), 0.7, 0.4, facecolor='#2ECC71', edgecolor='black', linewidth=1.5, alpha=0.7)
    ax.add_patch(rect_ff)
    ax.text(0.55, y_pos + 0.3, 'Feed\\nForward', ha='center', va='center', fontsize=7, fontweight='bold', color='white')
    
    # Add & Norm
    ax.text(1.1, y_pos - 0.1, '+ & Norm', ha='left', fontsize=7, style='italic')
    ax.text(1.1, y_pos + 0.5, '+ & Norm', ha='left', fontsize=7, style='italic')
    
    # Connections
    if i > 0:
        ax.arrow(1, y_pos - 1.7, 0, 0.1, head_width=0.1, head_length=0.1, fc='blue', ec='blue', linewidth=1.5)

# Encoder Output
rect_out = Rectangle((0, encoder_y + 4.5), 2, 1, facecolor='#F39C12', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect_out)
ax.text(1, encoder_y + 5, 'Encoder\\nOutput', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Arrows
ax.arrow(1, encoder_y - 0.5, 0, 0.3, head_width=0.15, head_length=0.15, fc='black', ec='black', linewidth=2)
ax.arrow(1, encoder_y + 1.5, 0, 0.3, head_width=0.15, head_length=0.15, fc='black', ec='black', linewidth=2)
ax.arrow(1, encoder_y + 3.3, 0, 0.3, head_width=0.15, head_length=0.15, fc='black', ec='black', linewidth=2)
ax.arrow(1, encoder_y + 5.5, 0, 0.3, head_width=0.15, head_length=0.15, fc='black', ec='black', linewidth=2)

# Labels
ax.text(1, encoder_y - 2.5, 'Encoder', ha='center', fontsize=14, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black', alpha=0.8))

# Decoder (simplified for visualization)
decoder_x = 4.5
rect_dec = Rectangle((decoder_x, encoder_y - 0.5), 2, 6, fill=False, edgecolor='red', linewidth=3, linestyle='--')
ax.add_patch(rect_dec)
ax.text(decoder_x + 1, encoder_y + 2, 'Decoder\\n(Similar\\nStructure)', ha='center', va='center', 
       fontsize=11, fontweight='bold', style='italic')

# Output
rect_final = Rectangle((7.5, encoder_y + 2), 2, 1, facecolor='#E67E22', edgecolor='black', linewidth=2, alpha=0.7)
ax.add_patch(rect_final)
ax.text(8.5, encoder_y + 2.5, 'Output', ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Connection from encoder to decoder
ax.arrow(2, encoder_y + 5, 2, -2, head_width=0.2, head_length=0.2, fc='green', ec='green', linewidth=2.5, alpha=0.7)
ax.arrow(decoder_x + 2, encoder_y + 2, 1.5, 0, head_width=0.2, head_length=0.2, fc='orange', ec='orange', linewidth=2.5, alpha=0.7)

plt.tight_layout()
plt.show()

print("✅ Transformer architecture visualization created!")
print("\\n💡 Key Components:")
print("   - Input Embeddings: Convert tokens to vectors")
print("   - Positional Encoding: Add position information")
print("   - Self-Attention: Learn relationships between tokens")
print("   - Feed Forward: Process information")
print("   - Add & Norm: Residual connections and normalization")

## Step 3: Visualize Self-Attention Mechanism

The heart of Transformers - understanding how attention works!

In [ ]:
# Visualize Self-Attention Mechanism
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(-1, 10)
ax.set_ylim(-1, 8)
ax.axis('off')
ax.set_title('Self-Attention Mechanism\\nHow Each Word Attends to All Other Words', 
             fontsize=16, fontweight='bold', pad=20)

# Example sentence: "The engine is running well"
words = ['The', 'engine', 'is', 'running', 'well']
n_words = len(words)

# Input words (bottom)
word_y = 1
for i, word in enumerate(words):
    x_pos = i * 1.8 + 1
    circle = Circle((x_pos, word_y), 0.3, color='#3498DB', ec='black', lw=2)
    ax.add_patch(circle)
    ax.text(x_pos, word_y, word, ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    ax.text(x_pos, word_y - 0.6, f'w{i+1}', ha='center', fontsize=9, style='italic')

# Attention weights visualization (middle)
attention_y = 4
# Example attention matrix (simplified)
attention_matrix = np.array([
    [0.3, 0.4, 0.1, 0.1, 0.1],  # "The" attends to
    [0.2, 0.5, 0.1, 0.15, 0.05],  # "engine" attends to
    [0.1, 0.3, 0.3, 0.2, 0.1],  # "is" attends to
    [0.1, 0.2, 0.2, 0.4, 0.1],  # "running" attends to
    [0.05, 0.1, 0.1, 0.15, 0.6]  # "well" attends to
])

# Draw attention connections
for i in range(n_words):
    x_from = i * 1.8 + 1
    for j in range(n_words):
        x_to = j * 1.8 + 1
        weight = attention_matrix[i, j]
        if weight > 0.1:  # Only show significant connections
            color_intensity = weight
            ax.plot([x_from, x_to], [word_y + 0.3, attention_y - 0.3], 
                   color='red', alpha=color_intensity, linewidth=weight * 3)
            if weight > 0.3:  # Label strong connections
                mid_x = (x_from + x_to) / 2
                mid_y = (word_y + 0.3 + attention_y - 0.3) / 2
                ax.text(mid_x, mid_y, f'{weight:.2f}', ha='center', fontsize=7, 
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.7))

# Output (top) - weighted sum
output_y = 6.5
for i, word in enumerate(words):
    x_pos = i * 1.8 + 1
    circle = Circle((x_pos, output_y), 0.3, color='#2ECC71', ec='black', lw=2)
    ax.add_patch(circle)
    ax.text(x_pos, output_y, f'O{i+1}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    
    # Connection from attention to output
    ax.arrow(x_pos, attention_y + 0.3, 0, 1.9, head_width=0.1, head_length=0.15, 
            fc='green', ec='green', linewidth=1.5, alpha=0.6)

# Attention layer box
rect_att = Rectangle((0.2, attention_y - 0.5), 8.6, 1, fill=False, edgecolor='red', linewidth=2.5, linestyle='--')
ax.add_patch(rect_att)
ax.text(4.5, attention_y, 'Self-Attention Layer', ha='center', fontsize=12, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='red', alpha=0.8))

# Labels
ax.text(4.5, word_y - 1.2, 'Input Words', ha='center', fontsize=11, fontweight='bold')
ax.text(4.5, output_y + 0.6, 'Output (Context-Aware Representations)', ha='center', fontsize=11, fontweight='bold')

# Explanation box
explanation = "Each word attends to all words (including itself).\\nThicker lines = stronger attention.\\nExample: 'engine' strongly attends to 'running'."
ax.text(4.5, 0.2, explanation, ha='center', fontsize=10, style='italic',
       bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black', alpha=0.7))

plt.tight_layout()
plt.show()

print("✅ Self-attention visualization created!")
print("\\n💡 How Self-Attention Works:")
print("   1. Each word creates Query (Q), Key (K), Value (V) vectors")
print("   2. Attention = similarity between Q and K")
print("   3. Output = weighted sum of Values based on attention")
print("   4. Each word gets context from all other words!")

In [ ]:
# Create simple sentiment analysis dataset
# Easy examples for testing!
sentences = [
    # Positive
    "I love this product",
    "This is amazing",
    "Great quality",
    "Excellent service",
    "Very satisfied",
    "Perfect solution",
    "Outstanding performance",
    "Highly recommended",
    "Wonderful experience",
    "Best purchase ever",
    
    # Negative
    "I hate this",
    "Terrible quality",
    "Very disappointed",
    "Poor service",
    "Not worth it",
    "Waste of money",
    "Completely broken",
    "Awful experience",
    "Worst product",
    "Do not buy",
    
    # Neutral
    "It is okay",
    "Average product",
    "Nothing special",
    "Just fine",
    "Acceptable quality"
]

labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # Positive
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  # Negative
          0.5, 0.5, 0.5, 0.5, 0.5]  # Neutral (we'll treat as negative for binary)

# Convert to binary (positive = 1, negative/neutral = 0)
labels_binary = [1 if l == 1 else 0 for l in labels]

print("✅ Simple sentiment dataset created!")
print(f"\\n📊 Dataset Statistics:")
print(f"   - Total sentences: {len(sentences)}")
print(f"   - Positive: {sum(labels_binary)}")
print(f"   - Negative/Neutral: {len(labels_binary) - sum(labels_binary)}")
print(f"\\n📝 Sample sentences:")
for i in range(5):
    sentiment = "Positive" if labels_binary[i] == 1 else "Negative"
    print(f"   '{sentences[i]}' → {sentiment}")

## Step 5: Prepare Text Data for Transformer

In [ ]:
# Tokenize and prepare text data
max_words = 1000  # Vocabulary size
max_length = 10  # Maximum sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

# Convert sentences to sequences
sequences = tokenizer.texts_to_sequences(sentences)

# Pad sequences to same length
X_text = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# Convert labels to numpy array
y_text = np.array(labels_binary)

print("✅ Text data prepared!")
print(f"\\n📊 Data Shapes:")
print(f"   - X_text shape: {X_text.shape}")
print(f"   - y_text shape: {y_text.shape}")
print(f"   - Vocabulary size: {len(tokenizer.word_index)}")
print(f"\\n📝 Example:")
print(f"   Sentence: '{sentences[0]}'")
print(f"   Sequence: {sequences[0]}")
print(f"   Padded: {X_text[0]}")
print(f"   Label: {y_text[0]} ({'Positive' if y_text[0] == 1 else 'Negative'})")

# Split data
from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42, stratify=y_text
)

print(f"\\n✅ Data split:")
print(f"   - Training: {X_train_text.shape[0]} samples")
print(f"   - Testing: {X_test_text.shape[0]} samples")

## Step 6: Build Transformer Block for Sentiment Analysis

Let's create a simple Transformer encoder block!

In [ ]:
# Define Transformer Encoder Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Define Token and Position Embedding
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

print("✅ Transformer components defined!")
print("\\n💡 Components:")
print("   - TransformerBlock: Self-attention + Feed-forward")
print("   - TokenAndPositionEmbedding: Word + Position embeddings")

## Step 7: Build and Train Transformer for Sentiment Analysis

In [ ]:
# Build Transformer model for sentiment analysis
embed_dim = 32  # Embedding dimension
num_heads = 2   # Number of attention heads
ff_dim = 32     # Feed-forward dimension

inputs = layers.Input(shape=(max_length,))
embedding_layer = TokenAndPositionEmbedding(max_length, max_words, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(20, activation="relu")(x)
x = Dropout(0.1)(x)
outputs = Dense(1, activation="sigmoid")(x)

model_transformer = Model(inputs=inputs, outputs=outputs)

model_transformer.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("📊 Transformer Model for Sentiment Analysis:")
print("=" * 60)
model_transformer.summary()

print("\\n💡 Architecture:")
print("   - Input: Tokenized sentences")
print("   - Embedding: Word + Position embeddings")
print("   - Transformer Block: Self-attention + Feed-forward")
print("   - Global Pooling: Aggregate sequence")
print("   - Output: Binary classification (positive/negative)")

In [ ]:
# Train the Transformer model
print("🚀 Training Transformer for Sentiment Analysis...")
print("=" * 60)

history_transformer = model_transformer.fit(
    X_train_text, y_train_text,
    batch_size=8,
    epochs=50,
    validation_data=(X_test_text, y_test_text),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

## Step 8: Test the Transformer Model

Let's test it with new sentences!

In [ ]:
# Test the model with new sentences
test_sentences = [
    "I really like this",
    "This is terrible",
    "Good product",
    "Not good at all",
    "Amazing quality",
    "Poor service",
    "Love it",
    "Hate it"
]

print("🧪 Testing Transformer Model:")
print("=" * 60)

for sentence in test_sentences:
    # Preprocess
    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
    
    # Predict
    prediction = model_transformer.predict(padded, verbose=0)[0][0]
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    
    print(f"'{sentence}'")
    print(f"  → {sentiment} (confidence: {confidence:.2%})")
    print()

# Evaluate on test set
test_loss, test_accuracy = model_transformer.evaluate(X_test_text, y_test_text, verbose=0)
print(f"\\n📊 Test Set Performance:")
print(f"   - Accuracy: {test_accuracy:.2%}")
print(f"   - Loss: {test_loss:.4f}")

## Step 9: Visualize Training Progress

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transformer Training Progress - Sentiment Analysis', fontsize=16, fontweight='bold')

# Loss
axes[0].plot(history_transformer.history['loss'], label='Train Loss', linewidth=2, color='blue')
axes[0].plot(history_transformer.history['val_loss'], label='Val Loss', linewidth=2, color='red', linestyle='--')
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history_transformer.history['accuracy'], label='Train Accuracy', linewidth=2, color='blue')
axes[1].plot(history_transformer.history['val_accuracy'], label='Val Accuracy', linewidth=2, color='red', linestyle='--')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Training progress visualization created!")

## Step 10: Visualize Attention Weights (Conceptual)

Let's visualize how attention works in our model!

In [ ]:
# Visualize attention weights (conceptual example)
example_sentence = "I love this amazing product"
words_example = example_sentence.split()

# Create a conceptual attention matrix
# In real transformers, this would come from the model
attention_matrix_example = np.array([
    [0.2, 0.3, 0.2, 0.15, 0.15],  # "I" attends to
    [0.1, 0.4, 0.3, 0.1, 0.1],    # "love" attends to
    [0.15, 0.3, 0.25, 0.15, 0.15], # "this" attends to
    [0.1, 0.2, 0.2, 0.3, 0.2],    # "amazing" attends to
    [0.1, 0.2, 0.2, 0.25, 0.25]   # "product" attends to
])

# Plot attention heatmap
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(attention_matrix_example, cmap='YlOrRd', aspect='auto')

# Set ticks
ax.set_xticks(np.arange(len(words_example)))
ax.set_yticks(np.arange(len(words_example)))
ax.set_xticklabels(words_example)
ax.set_yticklabels(words_example)

# Rotate labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
for i in range(len(words_example)):
    for j in range(len(words_example)):
        text = ax.text(j, i, f'{attention_matrix_example[i, j]:.2f}',
                      ha="center", va="center", color="black", fontweight='bold', fontsize=10)

ax.set_title('Self-Attention Weights\\n"How much each word attends to other words"', 
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Attended To (Key)', fontsize=12, fontweight='bold')
ax.set_ylabel('Attending From (Query)', fontsize=12, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Attention Weight', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Attention weights visualization created!")
print("\\n💡 Interpretation:")
print("   - Higher values (darker) = stronger attention")
print("   - Each row shows what one word attends to")
print("   - Example: 'love' strongly attends to 'amazing' and 'product'")
print("   - This helps the model understand context!")

## Step 11: Apply Transformer to RUL Prediction

Now let's use Transformers for time series RUL prediction!

In [ ]:
# Load NASA turbofan data for RUL prediction
from pathlib import Path

data_path = Path('dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

op_settings = ['Altitude', 'Mach', 'TRA']
sensors = [
    'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
    'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32'
]

def load_data(dataset='FD001'):
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print("✅ NASA turbofan data loaded!")
print(f"   - Training engines: {train_df['unit'].nunique()}")
print(f"   - Test engines: {test_df['unit'].nunique()}")

## Step 12: Create Sequences for Transformer RUL Prediction

In [ ]:
# Create sequences for Transformer
def create_sequences(data, sequence_length=30):
    sequences = []
    targets = []
    feature_cols = op_settings + sensors
    
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])
    
    return np.array(sequences), np.array(targets)

sequence_length = 30
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

# Scale features
from sklearn.preprocessing import StandardScaler
n_samples, n_timesteps, n_features = X_train_seq.shape
X_reshaped = X_train_seq.reshape(-1, n_features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)
X_train_seq_scaled = X_scaled.reshape(n_samples, n_timesteps, n_features)

# Split data
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_seq_scaled, y_train_seq, test_size=0.2, random_state=42
)

print("✅ Sequences created for Transformer RUL prediction!")
print(f"   - Training sequences: {X_train_split.shape[0]:,}")
print(f"   - Sequence length: {n_timesteps}")
print(f"   - Features: {n_features}")

## Step 13: Build Transformer for RUL Prediction

In [ ]:
# Build Transformer for RUL prediction (time series)
embed_dim_rul = 64
num_heads_rul = 4
ff_dim_rul = 128

# Input layer
inputs_rul = layers.Input(shape=(n_timesteps, n_features))

# Dense projection (since we have continuous values, not tokens)
x = Dense(embed_dim_rul)(inputs_rul)

# Positional encoding (learned)
positions = tf.range(start=0, limit=n_timesteps, delta=1)
position_embedding = Embedding(input_dim=n_timesteps, output_dim=embed_dim_rul)(positions)
x = x + position_embedding

# Transformer blocks
x = TransformerBlock(embed_dim_rul, num_heads_rul, ff_dim_rul)(x)
x = TransformerBlock(embed_dim_rul, num_heads_rul, ff_dim_rul)(x)

# Global pooling and output
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
outputs_rul = Dense(1)(x)  # Regression output

model_transformer_rul = Model(inputs=inputs_rul, outputs=outputs_rul)

model_transformer_rul.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("📊 Transformer Model for RUL Prediction:")
print("=" * 60)
model_transformer_rul.summary()

print("\\n💡 Architecture:")
print("   - Input: Sensor sequences (time series)")
print("   - Dense projection: Convert features to embedding dimension")
print("   - Positional encoding: Add time position information")
print("   - Transformer blocks: Self-attention for temporal patterns")
print("   - Output: RUL prediction (regression)")

In [ ]:
# Train Transformer for RUL prediction
print("🚀 Training Transformer for RUL Prediction...")
print("=" * 60)

history_transformer_rul = model_transformer_rul.fit(
    X_train_split, y_train_split,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_split, y_val_split),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)

print("\\n✅ Training completed!")

In [ ]:
# Evaluate Transformer RUL model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_transformer_rul = model_transformer_rul.predict(X_val_split, verbose=0)

mse_transformer = mean_squared_error(y_val_split, y_pred_transformer_rul)
mae_transformer = mean_absolute_error(y_val_split, y_pred_transformer_rul)
rmse_transformer = np.sqrt(mse_transformer)
r2_transformer = r2_score(y_val_split, y_pred_transformer_rul)

print("📊 Transformer RUL Prediction Performance:")
print("=" * 60)
print(f"MSE:  {mse_transformer:.2f}")
print(f"MAE:  {mae_transformer:.2f} cycles")
print(f"RMSE: {rmse_transformer:.2f} cycles")
print(f"R²:   {r2_transformer:.4f}")

print(f"\\n💡 Interpretation:")
print(f"   - Average error: {mae_transformer:.2f} cycles")
print(f"   - Model explains {r2_transformer*100:.1f}% of variance")

## Step 14: Visualize RUL Predictions

In [ ]:
# Plot RUL predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Transformer RUL Predictions', fontsize=16, fontweight='bold')

# Scatter plot
axes[0].scatter(y_val_split, y_pred_transformer_rul, alpha=0.5, s=20, color='blue')
min_val = min(y_val_split.min(), y_pred_transformer_rul.min())
max_val = max(y_val_split.max(), y_pred_transformer_rul.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted RUL (cycles)', fontsize=12, fontweight='bold')
axes[0].set_title(f'Predictions vs Actual (R² = {r2_transformer:.4f})', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training history
axes[1].plot(history_transformer_rul.history['loss'], label='Train Loss', linewidth=2, color='blue')
axes[1].plot(history_transformer_rul.history['val_loss'], label='Val Loss', linewidth=2, color='red', linestyle='--')
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Loss (MSE)', fontsize=12, fontweight='bold')
axes[1].set_title('Training Progress', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ RUL predictions visualization created!")

## 🎓 Summary and Key Takeaways

### ✅ What You've Learned:

1. **Transformer Architecture**:
   - Self-Attention: Each position attends to all positions
   - Multi-Head Attention: Multiple attention mechanisms
   - Positional Encoding: Adds position information
   - Feed-Forward Networks: Process attended information
   - Residual Connections: Helps with training

2. **Applications**:
   - **Language Tasks**: Sentiment analysis, text classification
   - **Time Series**: RUL prediction, forecasting
   - **Flexibility**: Works for both sequences and time series

3. **Key Advantages**:
   - **Parallel Processing**: Faster than RNNs
   - **Long-range Dependencies**: Better context understanding
   - **Attention Visualization**: Can see what model focuses on
   - **State-of-the-art**: Powers modern AI systems

### 💡 Important Insights:

- **Attention is Key**: Self-attention allows model to focus on important parts
- **Position Matters**: Positional encoding adds temporal/positional information
- **Multi-Head**: Multiple attention heads capture different relationships
- **Flexibility**: Same architecture works for language and time series

### 📚 Transformer vs RNN/LSTM:

| Aspect | RNN/LSTM | Transformer |
|--------|----------|-------------|
| **Processing** | Sequential | Parallel |
| **Speed** | Slower | Faster |
| **Long-range** | Limited | Excellent |
| **Attention** | Implicit | Explicit |
| **Context** | Local | Global |

### 🔧 Best Practices:

1. **Embedding Dimension**: Start with 32-128
2. **Attention Heads**: 2-8 heads typically work well
3. **Positional Encoding**: Essential for sequence understanding
4. **Dropout**: Prevents overfitting
5. **Learning Rate**: Lower learning rates often better

### 🆚 When to Use Transformers:

- ✅ Long sequences with important long-range dependencies
- ✅ When you need parallel processing
- ✅ When attention visualization is useful
- ✅ State-of-the-art performance needed
- ❌ Very short sequences (overkill)
- ❌ Limited computational resources

---

**Great job learning Transformers! 🔄🧠✨**